# Lab 03: How to split documents into chunks that keep their facts

**CPU | API key: `OPENAI_API_KEY` (embedding and chat calls) | In class**

## Goal

Load source files into records that say where each piece of text came from, and check what the parser handed over
before anything is searched. One moment teaches it. A one-page PDF holds a small table, and the parser gives the table
back as loose lines: the row *Boreal Mobile app 10 minutes* keeps its words and loses its header. Asked how long a
Boreal reset link is valid, the search ranks that row below the header line, which has no number in it. Written as one
sentence with its header, the same row ranks first. **The parser decides what the search can find.** The self-study
parts then cut text into chunks and measure what a cut can lose.

## What you will learn

- How to load a Markdown page and a PDF, page by page, into records that keep their file, heading or page and their
  character offsets (their lineage), and check that each policy sentence came through word for word.
- How a table read out of a PDF loses its header, and how writing each row as a sentence with its header changes what
  a search finds.
- How to cut a document into sentences or token windows, keep each chunk's source ID and offsets, and check that no
  text was lost.
- How to split a Markdown handbook at its headings and blank lines, and how to split by structure first under one
  token budget.
- How to compare chunkings on hit@3, tokens per question and answers to questions that cross a boundary.
- How a title header on each chunk changes what a search finds.

**In class:** Part A up to Checkpoint A. The word-for-word check at the end of Part A and Parts B to G are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-26
OPENAI_API_KEY    found in .env


Part A makes embedding calls only; the self-study parts make about 120 short chat calls, well under one US cent on
`gpt-4o-mini`. The saved outputs come from `gpt-4o-mini`; your run will differ a little, so compare the order of
results and the kind of reply with the text, not every number.

---

## Part A: Load a Markdown page and a PDF

Part A loads two source files into records and checks what came out before anything is searched. Nothing in it
needs a later part. Both files are in `resources/source-documents/` and were written for this lab:

- `atlas-help.md`, an Atlas help page: a title, one line of introduction, then four `## ` sections of two sentences each.
- `password-reset-comparison.pdf`, a one-page sheet that compares password resets in Atlas and Boreal.

Every fact in them restates the five Atlas and Boreal policies of `data/support-case.json`, text you already know.

### Load the help page as records

A Markdown file is already text: each line that starts with `## ` opens a section, which runs to the next heading.
A piece of text cannot say where it came from, so keep that beside it. A **record** here is a small dictionary:

- `chunk_id`: the record's own ID, the file's name plus a section number.
- `file` and `heading`: where the text came from, the record's **lineage** (also called **provenance metadata**).
- `start` and `end`: the **character offsets** of the text in the file, `start` included and `end` excluded, so
  `text[start:end]` gives the text back.
- `text`: the section itself.

`re.finditer(r"^## ", text, re.M)` finds every `## ` at the start of a line (`^` means "start of a line" because of
`re.M`); each match's `.start()` is its position. `markdown_records()` does the whole split for any Markdown file:

In [3]:
import re

# markdown_records(): one record per `## ` section of a Markdown file, with its heading and offsets
SOURCES = ROOT / "03_Chunking_With_Lineage" / "resources" / "source-documents"

def markdown_records(path):
    """One record per `## ` section: ID, file, heading, offsets in the file, and the text."""
    text = path.read_text(encoding="utf-8")
    # starts: the character where each "## " heading begins, plus the end of the file to close the last section
    starts = [match.start() for match in re.finditer(r"^## ", text, re.M)] + [len(text)]
    records = []
    for n in range(len(starts) - 1):
        start, end = starts[n], starts[n + 1]          # a section runs from its heading to the next one
        section = text[start:end]
        heading = section.split("\n")[0][3:]           # the first line of the section, without its "## "
        records.append({"chunk_id": f"{path.stem}-S{n}", "file": path.name, "heading": heading,
                        "start": start, "end": end, "text": section})
    return records

# help_records: one record per "## " section of the help page
help_records = markdown_records(SOURCES / "atlas-help.md")
for r in help_records:
    print(f"{r['chunk_id']:14} {r['start']:3} to {r['end']:3} | {r['heading']}")
print("\nthe first record's text:", repr(help_records[0]["text"]))

atlas-help-S0  158 to 274 | How do I reset my password?
atlas-help-S1  274 to 403 | How long does a reset link work?
atlas-help-S2  403 to 515 | Can I export my reports?
atlas-help-S3  515 to 628 | Where are my invoices?

the first record's text: '## How do I reset my password?\n\nAtlas passwords can be reset by email. Temporary accounts require a support agent.\n\n'


Four records, `atlas-help-S0` (characters 158 to 274) to `atlas-help-S3` (515 to 628), each with its heading. The
first record's text is its heading, a blank line, two sentences and a blank line. It starts at character 158, not 0:
the title and the introduction above the first `## ` belong to no section, so they are left out. **A split keeps only
what its rule names.**

### What happens step by step

1. `starts` holds the position of every heading, plus `len(text)` so the last section has an end.
2. The loop takes each start with the next one: that is one section.
3. `section.split("\n")[0]` is the heading line; `[3:]` drops its `## `.
4. Each record stores its ID (`path.stem` is the file name without `.md`), file, heading, offsets and text.

### Read the PDF page by page

A PDF is not text in the same way:

- It stores where each piece of text is drawn on a page, with no paragraphs, headings or tables.
- A parser rebuilds lines from those positions, so what you get depends on the parser.

We use [pypdf](https://pypdf.readthedocs.io/en/stable/user/extract-text.html), a PDF library in pure Python:
`PdfReader(path)` opens the file, `reader.pages` lists its pages, `page.extract_text()` returns one page's text. Read
the sheet page by page, keeping each page's number with its text:

In [4]:
from pypdf import PdfReader

# reader: the open PDF; reader.pages is its list of pages
reader = PdfReader(SOURCES / "password-reset-comparison.pdf")

# pdf_pages: one record per page, with the file, the page number and the text pypdf read from it
pdf_pages = []
for number, page in enumerate(reader.pages, 1):
    pdf_pages.append({"file": "password-reset-comparison.pdf", "page": number, "text": page.extract_text()})
print(len(pdf_pages), "page(s)\n")
print(pdf_pages[0]["text"])

1 page(s)

Password reset: Atlas and Boreal
Comparison sheet for the support team
Product Reset method Link valid for
Atlas Email 30 minutes
Boreal Mobile app 10 minutes
Boreal
Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
For Atlas, see the Atlas help page.


One page; `pdf_pages` is a list with one dictionary per page: `file`, `page` and `text`. The text reads in the order
of the sheet:

- Lines 1 and 2: the title and the subtitle.
- Lines 3 to 5: the table. The header *Product Reset method Link valid for* is a line of its own, then one line per
  row. Nothing marks where one column ends and the next begins.
- Lines 6 to 8: a *Boreal* heading, its sentence, and a pointer to the Atlas help page.

The row *Boreal Mobile app 10 minutes* kept its words and lost its header: nothing on that line says what *10 minutes*
is. The page has no headings to split at, so split it into the lines the parser gave, each with its page and offsets.
`re.finditer(r".+", text)` finds every run of characters up to a line break: one match per line.

In [5]:
# line_records(): one record per line of a page's text, with the page number and the line's offsets
def line_records(page):
    """One record per line of a page's text: ID, file, page, offsets in the page text, and the line."""
    stem = Path(page["file"]).stem                      # the file name without .pdf
    records = []
    for n, match in enumerate(re.finditer(r".+", page["text"])):
        records.append({"chunk_id": f"{stem}-p{page['page']}-L{n}", "file": page["file"], "page": page["page"],
                        "start": match.start(), "end": match.end(), "text": match.group()})
    return records

# sheet_lines: the line records of every page of the sheet (one page here)
sheet_lines = []
for page in pdf_pages:
    sheet_lines += line_records(page)
for r in sheet_lines:
    print(f"{r['chunk_id']:36} {r['start']:3} to {r['end']:3} | {r['text']}")

password-reset-comparison-p1-L0        0 to  32 | Password reset: Atlas and Boreal
password-reset-comparison-p1-L1       33 to  70 | Comparison sheet for the support team
password-reset-comparison-p1-L2       71 to 106 | Product Reset method Link valid for
password-reset-comparison-p1-L3      107 to 129 | Atlas Email 30 minutes
password-reset-comparison-p1-L4      130 to 158 | Boreal Mobile app 10 minutes
password-reset-comparison-p1-L5      159 to 165 | Boreal
password-reset-comparison-p1-L6      166 to 243 | Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
password-reset-comparison-p1-L7      244 to 279 | For Atlas, see the Atlas help page.


Eight line records, with the same fields as the help page's records but `page` in place of `heading`; the offsets
are in the page's text. The header is line `L2` (characters 71 to 106) and the Boreal row is line `L4` (130 to 158).

### What happens step by step

1. `Path(page["file"]).stem` is the file name without `.pdf`, used in each ID.
2. For each line `re.finditer` returns, `match.start()` and `match.end()` are its offsets and `match.group()` its text.
3. `sheet_lines` collects the records of every page.

### Search the records

Now search the 12 records of both files for a question the table answers. Two functions are Lab 01's, from
`shared/search.py`:

- `chunk_vectors(records, settings)` embeds each record's text: one vector per record, in the same order (saved in
  `.cache/`, so a second call costs nothing).
- `search(question, records, vectors, settings, k)` embeds the question, scores it against every vector and returns
  the best `k` as `(score, record)` pairs, best first.

No chat model is called in this part. Embed the twelve records first:

In [6]:
from shared.search import chunk_vectors, search

# records: the help page's 4 sections and the sheet's 8 lines, searched together
records = help_records + sheet_lines
# vectors: one embedding per record, row i for records[i]
vectors = chunk_vectors(records, settings)
print("vectors:", vectors.shape)

vectors: (12, 1536)


12 rows, one vector per record, and 1,536 numbers in each, the embedding model's length. The question:

> "How long is a Boreal reset link valid?"

*Predict first:* in which place will the line *Boreal Mobile app 10 minutes* rank, and which records will rank above
it?

In [7]:
# QUESTION: the question the table answers; the loop prints the five closest records
QUESTION = "How long is a Boreal reset link valid?"

# the top 5 records for QUESTION: rank, score, ID and the start of the text (line breaks turned into spaces)
for rank, (score, r) in enumerate(search(QUESTION, records, vectors, settings, k=5), 1):
    print(f"{rank}  {score:.4f}  {r['chunk_id']:36} {' '.join(r['text'].split())[:60]}")

1  0.6760  password-reset-comparison-p1-L6      Boreal reset links expire after 10 minutes. Password reset u
2  0.6521  atlas-help-S1                        ## How long does a reset link work? Atlas reset links expire
3  0.5872  password-reset-comparison-p1-L2      Product Reset method Link valid for
4  0.5185  password-reset-comparison-p1-L0      Password reset: Atlas and Boreal
5  0.4928  password-reset-comparison-p1-L4      Boreal Mobile app 10 minutes


The row that answers the question, *Boreal Mobile app 10 minutes*, ranks fifth (about 0.49). Above it:

1. the sheet's Boreal sentence, which also says *10 minutes* (about 0.68);
2. the help page's section on Atlas reset links, which says *30 minutes*;
3. the header line, which says what the number means but holds no number;
4. the sheet's title.

The parser kept every word of the row and lost the link between *10 minutes* and *Link valid for*; the answer is
still found only because the sheet also says it in a sentence. To write the row as a sentence with its header you need
its cells, and the plain text lost where a cell ends: is the method *Mobile* or *Mobile app*? pypdf's second mode,
`extraction_mode="layout"`, places the text as it sits on the page and fills the gaps between columns with spaces:

In [8]:
# layout: the same page, with the columns kept apart by runs of spaces
layout = reader.pages[0].extract_text(extraction_mode="layout")
print(layout)

Password reset: Atlas and Boreal
Comparison sheet for the support team


Product                    Reset method                             Link valid for

Atlas                      Email                                    30 minutes

Boreal                     Mobile app                               10 minutes


Boreal
Boreal reset links expire after 10 minutes. Password reset uses a mobile app.

For Atlas, see the Atlas help page.


### Write each table row as a sentence

The table now has its columns: two or more spaces between *Boreal*, *Mobile app* and *10 minutes*, and a single space
inside *Mobile app*. That gap is the rule for finding the cells:

- `re.search(r"\S\s{2,}\S", line)` is true for a line with a non-space, two or more spaces, then a non-space: a table
  line.
- `re.split(r"\s{2,}", line)` cuts a line wherever two or more spaces run: one string per cell.

In [9]:
# table_lines: the layout lines with a gap of two or more spaces between words
table_lines = [line for line in layout.split("\n") if re.search(r"\S\s{2,}\S", line)]
# cells: each table line cut at its gaps, one list of strings per line
cells = [re.split(r"\s{2,}", line.strip()) for line in table_lines]
header = cells[0]     # the first table line names the columns
rows = cells[1:]      # the other lines are the table's rows
print("header:", header)
print("rows:  ", rows)

header: ['Product', 'Reset method', 'Link valid for']
rows:   [['Atlas', 'Email', '30 minutes'], ['Boreal', 'Mobile app', '10 minutes']]


`header` is a list of three column names; `rows` is a list of two lists, one per row, three cells each. Now write
each row as one sentence that names every cell with its column: *Product: Boreal. Reset method: Mobile app. Link valid
for: 10 minutes.* This is **table-to-text** (also called table serialisation), the usual way to put a table into a
search index: one row at a time, with its column names. The `assert` checks that nothing was invented: a row's cells,
joined by single spaces, must be a line of page 1's plain text.

In [10]:
# written: for each row, the line as pypdf's plain text gave it -> the row as one sentence with its column names
written = {}
for row in rows:
    line = " ".join(row)                           # the row as a plain-text line, cells joined by single spaces
    assert line in pdf_pages[0]["text"], line      # the cells must be words the page holds
    written[line] = ". ".join(f"{name}: {cell}" for name, cell in zip(header, row)) + "."
    print(f"{line!r:32} -> {written[line]}")

'Atlas Email 30 minutes'         -> Product: Atlas. Reset method: Email. Link valid for: 30 minutes.
'Boreal Mobile app 10 minutes'   -> Product: Boreal. Reset method: Mobile app. Link valid for: 10 minutes.


Both rows passed the check, and the Boreal row now reads *Product: Boreal. Reset method: Mobile app. Link valid for:
10 minutes.*

### What happens step by step

1. `" ".join(row)` puts the cells back into the line pypdf printed; the `assert` stops the cell if that line is not
   on the page.
2. `zip(header, row)` pairs each column name with the row's cell, and each pair becomes `name: cell`.
3. `". ".join(...) + "."` joins the pairs into one sentence; `written` maps the plain-text line to it.

Put the sentences into the sheet's records. A written row keeps the page and offsets of the line it came from, but its
text is ours, which is why the check was needed. The header line comes out, since every row now carries it:

In [11]:
# sheet_rows: the sheet's line records, each table row's text swapped for its sentence, the header line left out
sheet_rows = []
for r in sheet_lines:
    if r["text"] == " ".join(header):
        continue
    sheet_rows.append({**r, "text": written.get(r["text"], r["text"])})   # a copy of the record with the new text
for r in sheet_rows:
    print(f"{r['chunk_id']:36} {r['start']:3} to {r['end']:3} | {r['text']}")

password-reset-comparison-p1-L0        0 to  32 | Password reset: Atlas and Boreal
password-reset-comparison-p1-L1       33 to  70 | Comparison sheet for the support team
password-reset-comparison-p1-L3      107 to 129 | Product: Atlas. Reset method: Email. Link valid for: 30 minutes.
password-reset-comparison-p1-L4      130 to 158 | Product: Boreal. Reset method: Mobile app. Link valid for: 10 minutes.
password-reset-comparison-p1-L5      159 to 165 | Boreal
password-reset-comparison-p1-L6      166 to 243 | Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
password-reset-comparison-p1-L7      244 to 279 | For Atlas, see the Atlas help page.


Seven records: the two rows as sentences, the other lines unchanged, no header line. `{**r, "text": ...}` is a copy
of the record with a new `text`; `written.get(text, text)` gives a row its sentence and any other line its old text.
Search the same question against the help page and these records:

In [12]:
# vectors_rows: one embedding per record of the help page plus the rewritten sheet
vectors_rows = chunk_vectors(help_records + sheet_rows, settings)
for rank, (score, r) in enumerate(search(QUESTION, help_records + sheet_rows, vectors_rows, settings, k=5), 1):
    print(f"{rank}  {score:.4f}  {r['chunk_id']:36} {' '.join(r['text'].split())[:60]}")

1  0.7109  password-reset-comparison-p1-L4      Product: Boreal. Reset method: Mobile app. Link valid for: 1
2  0.6760  password-reset-comparison-p1-L6      Boreal reset links expire after 10 minutes. Password reset u
3  0.6521  atlas-help-S1                        ## How long does a reset link work? Atlas reset links expire
4  0.5185  password-reset-comparison-p1-L0      Password reset: Atlas and Boreal
5  0.4869  password-reset-comparison-p1-L3      Product: Atlas. Reset method: Email. Link valid for: 30 minu


Written as a sentence with its header, the Boreal row ranks first (about 0.71), above the sheet's own sentence (about
0.68); the Atlas row, with *30 minutes*, is fifth. The cells did not change. **The row got back the words that say
what *10 minutes* is, and the search found it.**

Layout mode worked here because the table is small and evenly spaced. For merged cells, wrapped text or rows over
several pages, a library that reads the table's cell boxes does this step. Parts B to G (self-study)
go back to the policies stored as JSON and look at the next step: cutting text into chunks, and what a cut can lose.

> **Lesson to remember.** A parser keeps the words and can lose what binds them: a header to its rows, a number to its
> label. The search ranks only what the parser handed over. Whatever the tool, print what came out, find the sentences
> you know are in the file, and look for numbers that lost their labels.

### Checkpoint A

| You should see | Our run |
|---|---|
| The help page | 4 records, `atlas-help-S0` 158 to 274 up to `atlas-help-S3` 515 to 628 |
| The sheet | 1 page; the table as 3 lines; 8 line records, the header `L2`, the Boreal row `L4` |
| Search, lines as the parser gave them | `vectors: (12, 1536)`; *Boreal Mobile app 10 minutes* fifth (0.4928), the header line third |
| Rows as sentences | `header` `['Product', 'Reset method', 'Link valid for']`; both rows pass the check; 7 sheet records |
| Search, rows as sentences | the Boreal row first (0.7109), the sheet's sentence second |

Scores can move by a few thousandths between runs; check the order.

### Check the policy sentences word for word (self-study)

You know which sentences the two files should hold: those of the five policies in `data/support-case.json`. This
check finds each one in the help page, word for word, and names the record that holds it. Two library calls:

- `re.split(r"(?<=\.) ", text)` splits a policy into sentences at a space that follows a full stop (`(?<=\.)` means
  "a full stop just before").
- `text.find(sentence)` returns the character where the sentence starts in the page, or `-1` when it is not there.

`locate()` then returns the record whose offsets hold that position:

In [13]:
import json

# policies: the five Atlas and Boreal policies, each with a source_id and its two-sentence text
policies = json.loads((ROOT / "data" / "support-case.json").read_text(encoding="utf-8"))
# help_page: the whole help page as one string, the text the records' offsets point into
help_page = (SOURCES / "atlas-help.md").read_text(encoding="utf-8")

def locate(sentence, records, text):
    """The record whose offsets hold the sentence word for word, and where the sentence starts; (None, -1) if none."""
    start = text.find(sentence)
    for r in records:
        if start >= 0 and r["start"] <= start and start + len(sentence) <= r["end"]:
            return r, start
    return None, -1

# one line per policy sentence: the record and offsets that hold it, or "not on this page"
for doc in policies:
    for sentence in re.split(r"(?<=\.) ", doc["text"]):
        record, start = locate(sentence, help_records, help_page)
        where = f"{record['chunk_id']}, characters {start} to {start + len(sentence)}" if record else "not on this page"
        print(f"{doc['source_id']}  {where:40} | {sentence}")

H01  atlas-help-S0, characters 190 to 228     | Atlas passwords can be reset by email.
H01  atlas-help-S0, characters 229 to 272     | Temporary accounts require a support agent.
H02  atlas-help-S1, characters 311 to 353     | Atlas reset links expire after 30 minutes.
H02  atlas-help-S1, characters 354 to 401     | Request a new link if the old link has expired.
H03  atlas-help-S2, characters 432 to 467     | Atlas exports reports as CSV files.
H03  atlas-help-S2, characters 468 to 513     | Only workspace owners can export all reports.
H04  not on this page                         | Boreal reset links expire after 10 minutes.
H04  not on this page                         | Password reset uses a mobile app.
H05  atlas-help-S3, characters 542 to 574     | Atlas invoices are sent monthly.
H05  atlas-help-S3, characters 575 to 627     | Workspace owners can download invoices from Billing.


Eight of the ten policy sentences are on the help page, word for word, each inside the section about its topic: H01's
two in `atlas-help-S0`, H05's two in `atlas-help-S3`. H04's two Boreal sentences are not on this page; the sheet should
hold them. A sentence that is missing, changed by one word, or cut by a section boundary would show as *not on this
page*. Check H04's two sentences against each page of the sheet:

In [14]:
# for each H04 sentence, the pages of the sheet whose text holds it word for word
for sentence in re.split(r"(?<=\.) ", policies[3]["text"]):
    print("found on page", [p["page"] for p in pdf_pages if sentence in p["text"]], "|", sentence)

found on page [1] | Boreal reset links expire after 10 minutes.
found on page [1] | Password reset uses a mobile app.


Both of H04's sentences are on page 1, word for word: pypdf kept the sentences as they were written and lost only the
table's structure.

| You should see | Our run |
|---|---|
| The help page | 8 of 10 policy sentences in it, each in its section (H01's two at characters 190 to 272); H04's 2 not on it |
| The sheet | H04's 2 sentences both found on page 1 |

---

## Part B: Split the policies into sentences (self-study)

Part B goes back to the five policies as stored in JSON, and shows what a chunk boundary can do to an answer, on
documents short enough to read whole.

A **chunk** is the piece of text a search ranks and an answer reads. Each chunk keeps its **lineage**: the ID of the
document it came from and the **character offsets** of its text in that document, `start` included and `end`
excluded, so `text[start:end]` gives the chunk back. Lab 01's chunks carry the same fields. Load the five policies:

In [15]:
import json

# DATA: the data folder; policies: the five policies as stored in JSON, each with a source_id, a title and its text
DATA = ROOT / "data"
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
for doc in policies:
    print(doc["source_id"], doc["title"], "|", doc["text"])

H01 Atlas reset | Atlas passwords can be reset by email. Temporary accounts require a support agent.
H02 Atlas expiry | Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
H03 Atlas export | Atlas exports reports as CSV files. Only workspace owners can export all reports.
H04 Boreal reset | Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
H05 Atlas billing | Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.


Each policy has two sentences. `chunk()` builds one chunk record from a document and two offsets. `sentences()` finds
the sentences with a **regular expression**: from a non-space character to the first `.`, `!` or `?` that is followed by
a space, or to the end of the text. `finditer` returns each match with its offsets.

In [16]:
import re

# chunk(): one chunk record from a document and two offsets
def chunk(doc, n, start, end):
    """Chunk n of a document: its own ID, the document's ID and title, and where its text sits in the document."""
    return {"chunk_id": f"{doc['source_id']}-C{n:02d}", "source_id": doc["source_id"], "title": doc["title"],
            "start": start, "end": end, "text": doc["text"][start:end]}

# SENTENCE: from a non-space character to the first . ! or ? followed by a space, or to the end of the text
SENTENCE = re.compile(r"\S.*?(?:[.!?](?=\s)|$)", re.S)

# sentences(): one chunk per sentence of a document, with its offsets
def sentences(doc):
    return [chunk(doc, n, match.start(), match.end()) for n, match in enumerate(SENTENCE.finditer(doc["text"]))]

for c in sentences(policies[0]):
    print(c)

{'chunk_id': 'H01-C00', 'source_id': 'H01', 'title': 'Atlas reset', 'start': 0, 'end': 38, 'text': 'Atlas passwords can be reset by email.'}
{'chunk_id': 'H01-C01', 'source_id': 'H01', 'title': 'Atlas reset', 'start': 39, 'end': 82, 'text': 'Temporary accounts require a support agent.'}


H01 becomes two chunks, `H01-C00` (characters 0 to 38) and `H01-C01` (39 to 82). Now two questions a customer might
ask, each with words a right answer contains and a word it must not contain. `ask()` embeds the chunks
(`chunk_vectors()`), searches (`search()`), packs the top 3 (`pack()`) and answers (`answer()`) five times per
question, then prints how many replies were right, the packed IDs and the first reply.

In [17]:
from shared.search import chunk_vectors, search
from shared.answer import answer, pack

POLICY_QUESTIONS = [   # (question, a right answer says, a right answer never says)
    ("How can I reset my Atlas password?", "email", "app"),
    ("How do I reset the password on a temporary Atlas account?", "support agent", "insufficient"),
]

# right(): a reply is right when it says the expected words and never says the forbidden one
def right(reply, says, never):
    return says in reply.lower() and never not in reply.lower()

# ask(): search, pack and answer each policy question `runs` times from a chunking; returns how many replies were right
def ask(chunks, runs=5):
    """Answer each policy question `runs` times from the top 3 chunks; print the results, return how many were right."""
    vectors = chunk_vectors(chunks, settings)
    total = 0
    for question, says, never in POLICY_QUESTIONS:
        lines = pack(search(question, chunks, vectors, settings, k=3), settings)
        replies = [answer(question, lines, settings)["answer"] for _ in range(runs)]
        total += sum(right(reply, says, never) for reply in replies)
        print(f"  right {sum(right(r, says, never) for r in replies)} of {runs} | packed",
              [line[1:line.index("]")] for line in lines], "|", replies[0])
    return total

# policy_sentences: the ten sentence chunks of the five policies
policy_sentences = [c for doc in policies for c in sentences(doc)]
print(len(policy_sentences), "sentence chunks")
print("right in all:", ask(policy_sentences), "of 10")

10 sentence chunks


  right 0 of 5 | packed ['H01-C00', 'H04-C01', 'H02-C00'] | Atlas passwords can be reset by email or through a mobile app.


  right 0 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C01'] | Insufficient evidence.
right in all: 0 of 10


### What happens step by step

1. `POLICY_QUESTIONS` holds each question with a word a right answer says and a word it never says.
2. `chunk_vectors()` embeds the chunks once; `search()` ranks them for the question and keeps the top 3.
3. `pack()` writes those three as `[ID] text` lines, the evidence the answer step reads.
4. `answer()` is called `runs` times, five here, since replies vary; `right()` scores each reply.
5. Each question prints its right count, the packed IDs and the first reply; the cell returns the total over both.

Both questions are wrong in all five runs. The first reply to the first question adds a mobile app to the Atlas reset; the first reply to the second is *Insufficient evidence.*, though H01 says what a temporary account needs. Before reading why, ask the same questions with each policy kept whole, one chunk per document:

In [18]:
# policy_whole: one chunk per policy, its whole text
policy_whole = [chunk(doc, 0, 0, len(doc["text"])) for doc in policies]
print("right in all:", ask(policy_whole), "of 10")

  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Temporary Atlas accounts require a support agent to reset the password.
right in all: 10 of 10


With whole policies both questions are right in 5 of 5 runs; the first replies say *by email* and *a support agent*. The documents and the answer step did not change; only the boundaries did. Read the five best
sentence chunks for each question:

In [19]:
# the five best sentence chunks for each question, with their scores
vectors = chunk_vectors(policy_sentences, settings)
for question, _, _ in POLICY_QUESTIONS:
    print(question)
    for score, c in search(question, policy_sentences, vectors, settings, k=5):
        print(f"  {score:.4f}  {c['chunk_id']}  {c['text']}")

How can I reset my Atlas password?


  0.6933  H01-C00  Atlas passwords can be reset by email.
  0.4851  H04-C01  Password reset uses a mobile app.
  0.4293  H02-C00  Atlas reset links expire after 30 minutes.
  0.3512  H04-C00  Boreal reset links expire after 10 minutes.
  0.2907  H01-C01  Temporary accounts require a support agent.
How do I reset the password on a temporary Atlas account?


  0.6541  H01-C00  Atlas passwords can be reset by email.
  0.4607  H02-C00  Atlas reset links expire after 30 minutes.
  0.4483  H04-C01  Password reset uses a mobile app.
  0.4449  H01-C01  Temporary accounts require a support agent.
  0.3789  H04-C00  Boreal reset links expire after 10 minutes.


For the first question, `H04-C01` comes second: *Password reset uses a mobile app.* It is Boreal's policy, but the
word *Boreal* was in the sentence before it, so the chunk reads like a general rule and the model adds it to the Atlas
answer. For the second, `H01-C01`, *Temporary accounts require a support agent.*, is fourth: it never says *Atlas*,
so three chunks that do say it rank higher, and the answer step never sees the exception. Both chunks are true and
both are in the index. Each lost something its document gave it: a subject, or the rule it qualifies.

### Checkpoint B

| You should see | Our run |
|---|---|
| Sentence chunks | 10; `H01-C00` 0 to 38, `H01-C01` 39 to 82 |
| Sentences, right answers | 0 of 5 and 0 of 5; first replies *Atlas passwords can be reset by email or through a mobile app.* and *Insufficient evidence.* |
| Whole policies, right answers | 5 of 5 and 5 of 5 |
| Ranks | `H04-C01` second for the first question; `H01-C01` fourth for the second |

---

## Part C: Cut token windows and check their offsets (self-study)

Part C builds the oldest chunker: cut every so many tokens, whatever the text says. Lab 01's 272 chunks were made
this way. Windows are a fundamental; Part E builds what replaces them in practice.

A **token window** is a run of tokens of fixed length; a new window starts every `step` tokens. Count tokens with
`tiktoken` and the encoding `TOKEN_ENCODING` names. To keep lineage, each token needs its position in the text:
`decode_with_offsets` returns the character where each token starts. Take a Tesla paragraph:

In [20]:
import tiktoken

# encoding: the tokenizer TOKEN_ENCODING names; count(): how many tokens a text has
encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])

def count(text):
    return len(encoding.encode(text))

# tesla: one Wikipedia paragraph; ids: its tokens; offsets: the character where each token starts
passages = json.loads((DATA / "squad-passages.json").read_text(encoding="utf-8"))
tesla = next(p for p in passages if p["source_id"] == "S03-P01")
ids = encoding.encode(tesla["text"])
_, offsets = encoding.decode_with_offsets(ids)
print(len(ids), "tokens,", len(tesla["text"]), "characters")
for token, offset in list(zip(ids, offsets))[:6]:
    print(f"token {token:6} starts at character {offset:3}: {encoding.decode([token])!r}")

114 tokens, 607 characters
token 181412 starts at character   0: 'Tesla'
token  27841 starts at character   5: ' gained'
token   3240 starts at character  12: ' experience'
token    306 starts at character  23: ' in'
token   5443 starts at character  26: ' tele'
token  45577 starts at character  31: 'phony'


114 tokens, 607 characters. *gained* starts at character 5 with the space before it: a token often carries that space, and a long word can be two tokens (*tele*, *phony*). A window from token `first` to token `last` (not included) runs from `offsets[first]` to the start
of the token after it, `offsets[last]`; one extra offset, the text's length, ends the last window. With the sizes Lab
01's chunks used, 96 tokens and a new window every 80:

In [21]:
# windows(): chunks of `size` tokens, a new one every `step` tokens, each with character offsets
def windows(doc, size=96, step=80):
    """Windows of `size` tokens, a new one every `step` tokens, as chunks with character offsets."""
    ids = encoding.encode(doc["text"])
    _, offsets = encoding.decode_with_offsets(ids)
    offsets.append(len(doc["text"]))
    chunks = []
    for n, first in enumerate(range(0, len(ids), step)):
        last = min(first + size, len(ids))
        chunks.append(chunk(doc, n, offsets[first], offsets[last]))
        if last == len(ids):
            break
    return chunks

for c in windows(tesla):
    print(c["chunk_id"], c["start"], c["end"], count(c["text"]), "tokens |", c["text"][:40], "...", c["text"][-35:])

S03-P01-C00 0 509 96 tokens | Tesla gained experience in telephony and ... power development was involved in a
S03-P01-C01 417 607 34 tokens |  consultant. His work in the formative y ...  as well as various patent battles.


Two windows. `S03-P01-C00` holds tokens 0 to 95, characters 0 to 509; `S03-P01-C01` starts at token 80, character 417, and holds the 34 tokens to the end. The first ends on *involved in a*; the second starts with the space before *consultant*. A window ends wherever its 96th token ends, in the middle of a sentence or a word. The windows share 16 tokens (characters 417 to 509): this **overlap** gives a fact cut at one boundary a second chance in the next window, and is stored, embedded and read twice.

Lineage is only useful if it is right. `lost_characters()` checks two things: that every chunk's text is the slice
its offsets name in its document, and how many non-space characters of the documents no chunk holds. Run it on the
windows of all 120 paragraphs, on the sentences of the same paragraphs, and on Lab 01's frozen 272 chunks.

In [22]:
# lost_characters(): the lineage check, every chunk's slice against its text, and the characters no chunk holds
def lost_characters(chunks, docs):
    """Assert each chunk is the slice its offsets name; return how many non-space characters no chunk holds."""
    texts = {doc["source_id"]: doc["text"] for doc in docs}
    kept = set()
    for c in chunks:
        assert texts[c["source_id"]][c["start"]:c["end"]] == c["text"], c["chunk_id"]
        kept.update((c["source_id"], i) for i in range(c["start"], c["end"]))
    return sum((source_id, i) not in kept and not character.isspace()
               for source_id, text in texts.items() for i, character in enumerate(text))

# frozen: Lab 01's 272 chunks; passage_windows and passage_sentences: the 120 paragraphs cut both ways
frozen = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
passage_windows = [c for p in passages for c in windows(p)]
passage_sentences = [c for p in passages for c in sentences(p)]
for name, chunks in [("windows", passage_windows), ("sentences", passage_sentences), ("frozen 272", frozen)]:
    print(f"{name:10} {len(chunks):4} chunks, lost characters: {lost_characters(chunks, passages)}")

windows     274 chunks, lost characters: 0
sentences   612 chunks, lost characters: 0
frozen 272  272 chunks, lost characters: 0


Every slice matches and nothing is lost, for the 274 windows, the 612 sentences and the frozen 272 chunks. The frozen chunks used the same rule with another model's tokenizer, so the rule gives 274 windows here,
not 272: a new tokenizer, size or step is a new chunking, embedded and measured again. The check cannot see a bad
boundary, such as the regular expression's cut after the initial in *Dwight D. Eisenhower*; it sees lost text and
wrong offsets.

### Checkpoint C

| You should see | Our run |
|---|---|
| The Tesla paragraph | 114 tokens, 607 characters; *Tesla* at 0, *gained* at 5 |
| Its windows | `S03-P01-C00` 0 to 509 (96 tokens), `S03-P01-C01` 417 to 607 (34 tokens) |
| Lineage check | windows 274, sentences 612, frozen 272; lost characters 0 for each |

---

## Part D: Split a Markdown handbook at its headings (self-study)

Part D splits a longer Markdown document at its headings, as Part A split the help page, then into paragraphs, and
measures both splits on labelled questions; the policies and the Wikipedia paragraphs have only sentences.

`resources/support_handbook.md` is the support handbook of Chinook, a made-up music shop: 20 policies, each under a
`##` heading and a `Source:` line naming its policy file. It is copied unchanged from another of the author's lab
repositories, so its first line names a build script that is not in this repo; the README gives the
file's SHA-256. Load it as one document:

In [23]:
# handbook: the whole handbook as one document with the ID HB
RESOURCES = ROOT / "03_Chunking_With_Lineage" / "resources"
handbook = {"source_id": "HB", "title": "Chinook support handbook",
            "text": (RESOURCES / "support_handbook.md").read_text(encoding="utf-8")}
print(count(handbook["text"]), "tokens,", len(handbook["text"]), "characters")
print(handbook["text"][:420])

1590 tokens, 7428 characters
# Chinook support handbook

Built from the 20 policy files in data/corpus/ by tools/D01/samples/s23_handbook_build_offline.py. Do not edit: rebuild it from the corpus.

## Accounts
Source: policy_accounts.md

An account is the email plus the customer record. One email, one account.

A customer can change their email once per calendar year. Invoices already issued keep the old email as a snapshot.

Account deletion: w


Every section starts on a line that begins with `## `. `sections()` finds those lines with a regular expression
(`re.M` makes `^` match at the start of every line, not only of the text) and cuts from each heading to the next with
Part B's `chunk()`, so every section keeps its offsets. It also reads the `Source:` line, so a citation can name the
policy file.

In [24]:
# HEADING: a "## " at the start of a line; SOURCE: the policy file a "Source:" line names
HEADING = re.compile(r"^## ", re.M)
SOURCE = re.compile(r"^Source: (\S+)", re.M)

# sections(): one chunk per section, from its heading to the next, with its policy file
def sections(doc):
    """One chunk per `## ` section, from its heading to the next one, with the policy file its Source line names."""
    starts = [match.start() for match in HEADING.finditer(doc["text"])] + [len(doc["text"])]
    found = []
    for n, (start, end) in enumerate(zip(starts, starts[1:])):
        c = chunk(doc, n, start, end)
        found.append({**c, "file": SOURCE.search(c["text"]).group(1)})
    return found

handbook_sections = sections(handbook)
print(len(handbook_sections), "sections | lost characters:", lost_characters(handbook_sections, [handbook]))
first = handbook_sections[0]
print(first["chunk_id"], first["file"], first["start"], first["end"], count(first["text"]), "tokens")
print(first["text"])

20 sections | lost characters: 145
HB-C00 policy_accounts.md 169 568 84 tokens
## Accounts
Source: policy_accounts.md

An account is the email plus the customer record. One email, one account.

A customer can change their email once per calendar year. Invoices already issued keep the old email as a snapshot.

Account deletion: we remove the login and the marketing flag. Invoices stay. Support tickets stay for 24 months, then they are dropped. See policy_data_retention.md.




20 sections, one per policy; `HB-C00` is Accounts, characters 169 to 568, 84 tokens. The 145 lost characters are
the title and build note above the first heading: the check shows exactly what a split leaves out.

Smaller pieces next. `paragraphs()` cuts each section at its blank lines, skips the heading and `Source:` line, and
keeps the heading as the chunk's `title` for Part G.

In [25]:
BLOCK = re.compile(r"(?:^.+\n?)+", re.M)       # one or more lines with text, up to a blank line

# paragraphs(): the blocks of a section between blank lines, as chunks with the heading as their title
def paragraphs(section):
    """The blocks of a section between blank lines, after its heading and Source line, as chunks of the handbook."""
    heading = section["text"].split("\n")[0][3:]
    blocks = list(BLOCK.finditer(section["text"]))[1:]
    return [{**chunk(handbook, 0, section["start"] + b.start(), section["start"] + b.end()),
             "chunk_id": f"{section['chunk_id']}-P{n}", "file": section["file"], "title": heading}
            for n, b in enumerate(blocks)]

handbook_paragraphs = [p for s in handbook_sections for p in paragraphs(s)]
print(len(handbook_paragraphs), "paragraphs | lost characters:", lost_characters(handbook_paragraphs, [handbook]))
for p in handbook_paragraphs:
    if p["file"] == "policy_escalations.md":
        print(p["chunk_id"], "|", p["text"].strip())

67 paragraphs | lost characters: 986
HB-C05-P0 | Escalate when:
HB-C05-P1 | - a refund over 100 dollars is requested
- the same customer has three open tickets
- a representative wants to override a written policy
HB-C05-P2 | Escalations go to the support manager. The assigned representative stays on the ticket.
HB-C05-P3 | This shop does not publish the manager's personal phone number. Do not invent one.


67 paragraphs. The lost characters now include the headings and `Source:` lines, left out on purpose: `file` keeps
the source, and Part G puts the heading back. In the escalation policy, *Escalate when:* is a chunk of its own, cut
from its list by a blank line, as Part B's sentence was cut from its product: the layout is not always the meaning.

`resources/handbook-questions.json` holds 24 customer questions, each labelled with its answering policy before any
search: 15 in a customer's words (*Can I pay with bitcoin?*), 9 with a rare word from the policy (*Is split tender
allowed?*). `measure_handbook()` finds, for each, the rank of the first answering chunk and the tokens of the top 3:

In [26]:
import numpy as np
from shared.search import embed
from shared.settings import embed_url

# handbook_questions: the 24 labelled questions; H: one vector per question
handbook_questions = json.loads((RESOURCES / "handbook-questions.json").read_text(encoding="utf-8"))["questions"]
H = embed([q["question"] for q in handbook_questions], settings)

# tokens(): the tokens in a list of chunks
def tokens(chunks):
    return sum(count(c["text"]) for c in chunks)

# measure_handbook(): per question, the rank of the first answering chunk; and the mean tokens of the top 3
def measure_handbook(chunks):
    """Per question, the rank of the first chunk from an answering file; and the mean tokens of the top 3."""
    ranks, cost = [], []
    for q, row in zip(handbook_questions, H @ chunk_vectors(chunks, settings).T):
        order = [chunks[i] for i in np.argsort(-row, kind="stable")]
        ranks.append(next(rank for rank, c in enumerate(order, 1) if c["file"] in q["files"]))
        cost.append(tokens(order[:3]))
    return ranks, np.mean(cost)

# report(): one line per chunking, then the questions whose answering chunk was not first
def report(name, chunks):
    ranks, cost = measure_handbook(chunks)
    print(f"{name:22} {len(chunks):3} chunks | right first {sum(r == 1 for r in ranks):2} of 24 |"
          f" in the top 3 {sum(r <= 3 for r in ranks):2} of 24 | {cost:4.0f} tokens per question")
    for q, r in zip(handbook_questions, ranks):
        if r > 1:
            print(f"    rank {r}: {q['question']}")

report("sections", handbook_sections)
report("paragraphs", handbook_paragraphs)
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

sections                20 chunks | right first 21 of 24 | in the top 3 24 of 24 |  241 tokens per question
    rank 3: When can I reach a person?
    rank 2: Can I swap the CD for the vinyl version?
    rank 2: What happens after the first byte of a download?


paragraphs              67 chunks | right first 23 of 24 | in the top 3 24 of 24 |   61 tokens per question
    rank 3: When can I reach a person?

text-embedding-3-small at https://api.openai.com/v1


### What happens step by step

1. `embed()` turns the 24 questions into the matrix `H`, one row per question.
2. `H @ chunk_vectors(chunks, settings).T` scores every question against every chunk; each row is one question.
3. `np.argsort(-row, kind="stable")` orders the chunks for that question, best first.
4. The first chunk whose `file` is one of the question's answering files gives the rank; `tokens(order[:3])` counts
   what the top 3 would cost the answer step.
5. `report()` prints how many questions had an answering chunk first and in the top 3, the mean tokens, then each
   question that was not first with its rank.

Both splits find an answering policy in the top 3 for all 24 questions. Paragraphs put it first more often, 23 of 24
against 21, at a quarter of the tokens (61 a question against 241): a section holds every rule of a policy, a
paragraph one. Each question here needs one rule, though; before you choose paragraphs for a real handbook, add
questions whose answer spans two paragraphs, like the escalation list.

### Checkpoint D

| You should see | Our run |
|---|---|
| The handbook | 1,590 tokens, 7,428 characters |
| Sections | 20; lost characters 145 (the title and build note); `HB-C00` Accounts, 169 to 568, 84 tokens |
| Paragraphs | 67; lost characters 986; *Escalate when:* alone in `HB-C05-P0` |
| The 24 questions | sections: first 21, top 3 24, 241 tokens a question; paragraphs: 23, 24, 61 |

---

## Part E: Split by structure first (self-study)

Part E builds the splitter current guides start from, **recursive splitting**: use the document's own boundaries,
largest first, and cut smaller only when a piece is over the budget. Part D split at headings; these documents have only sentences. With the windows' 96-token budget: keep the
whole document when it fits; else join its sentences in order while the chunk stays within the budget; cut only a
sentence longer than the budget into windows.

In [27]:
# structured(): the whole document if it fits the budget; else sentences joined up to it; windows for one that is too long
def structured(doc, budget=96):
    """The whole document if it fits; else sentences joined while they fit; windows for a sentence that is too long."""
    if count(doc["text"]) <= budget:
        return [chunk(doc, 0, 0, len(doc["text"]))]
    spans = []
    for s in sentences(doc):
        if spans and count(doc["text"][spans[-1][0]:s["end"]]) <= budget:
            spans[-1] = (spans[-1][0], s["end"])                   # the sentence joins the chunk before it
        elif count(s["text"]) <= budget:
            spans.append((s["start"], s["end"]))                   # the sentence starts a new chunk
        else:                                                      # one sentence over the budget: cut it into windows
            cut = windows({**doc, "text": s["text"]}, size=budget, step=budget)
            spans += [(s["start"] + w["start"], s["start"] + w["end"]) for w in cut]
    return [chunk(doc, n, start, end) for n, (start, end) in enumerate(spans)]

for c in structured(policies[0]) + structured(tesla):
    print(c["chunk_id"], c["start"], c["end"], count(c["text"]), "tokens |", c["text"][:40], "...", c["text"][-35:])

H01-C00 0 82 15 tokens | Atlas passwords can be reset by email. T ... y accounts require a support agent.
S03-P01-C00 0 429 82 tokens | Tesla gained experience in telephony and ... a for a short time as a consultant.
S03-P01-C01 430 607 32 tokens | His work in the formative years of elect ...  as well as various patent battles.


H01 fits the budget (15 tokens) and stays whole, the rule beside its exception. The Tesla paragraph (114 tokens) does not, so it becomes two chunks of whole sentences: 82 tokens ending *as a consultant.*, and 32 tokens starting *His work*. The windows cut the same paragraph inside a sentence. Run it on all 120 paragraphs and the five policies, check the lineage, and count how each document was
split:

In [28]:
# docs: the 120 paragraphs and the five policies; docs_structured: all of them split by structure; pieces: chunks per document
docs = passages + policies
docs_structured = [c for doc in docs for c in structured(doc)]
pieces = [len(structured(doc)) for doc in docs]
print(len(docs_structured), "chunks | lost characters:", lost_characters(docs_structured, docs))
print("documents kept whole:", pieces.count(1), "| split:", len(docs) - pieces.count(1),
      "| largest chunk:", max(count(c["text"]) for c in docs_structured), "tokens")

288 chunks | lost characters: 0
documents kept whole: 16 | split: 109 | largest chunk: 96 tokens


288 chunks, and nothing lost. At 96 tokens only 16 of the 125 documents fit whole, the five policies and 11 paragraphs; the other 109 were split, and no chunk is over the budget.

### Checkpoint E

| You should see | Our run |
|---|---|
| H01 and Tesla | `H01-C00` 0 to 82 (15 tokens); Tesla `S03-P01-C00` 0 to 429 (82 tokens) and `S03-P01-C01` 430 to 607 (32) |
| All documents | 288 chunks, lost characters 0; 16 kept whole, 109 split; largest chunk 96 tokens |

---

## Part F: Compare four chunkings (self-study)

Part F puts the four chunkings side by side on the same documents, the 120 paragraphs and the five policies, and
measures each in three ways:

- **hit@3** on the 30 development questions: does a top-3 chunk come from the question's paragraph?
- **tokens per question**: what those three chunks cost the answer step.
- **policy answers right**: Part B's two questions, five runs each.

`top3()` ranks every chunk of a chunking for each of the 30 development questions, embedded once, and keeps the best three:

In [29]:
# questions: the 30 development questions; Q: one vector per question
questions = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
Q = embed([q["question"] for q in questions], settings)

def top3(chunks):
    """The three best chunks for each development question."""
    scores = Q @ chunk_vectors(chunks, settings).T
    return [[chunks[i] for i in np.argsort(-row, kind="stable")[:3]] for row in scores]

`measure()` joins `top3()`, Part D's `tokens()` and `ask()` from Part B; `ask()` prints its two lines for each
chunking.

In [30]:
# measure(): for one chunking, its chunk count, hit@3, tokens per question and policy answers right
def measure(chunks):
    tops = top3(chunks)
    hits = sum(any(c["source_id"] in q["known_support"] for c in top) for q, top in zip(questions, tops))
    per_question = np.mean([tokens(top) for top in tops])
    return len(chunks), hits, per_question, ask(chunks)

# CHUNKINGS: the four chunkings of the same 125 documents
CHUNKINGS = {
    "passage": [chunk(doc, 0, 0, len(doc["text"])) for doc in docs],
    "window": [c for doc in docs for c in windows(doc)],
    "sentence": [c for doc in docs for c in sentences(doc)],
    "structured": docs_structured,
}
results = {}
for name, chunks in CHUNKINGS.items():
    print(name)
    results[name] = measure(chunks)

passage


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Temporary Atlas accounts require a support agent to reset the password.
window


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Temporary Atlas accounts require a support agent to reset the password.
sentence


  right 0 of 5 | packed ['H01-C00', 'H04-C01', 'H02-C00'] | Atlas passwords can be reset by email or through a mobile app.


  right 0 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C01'] | Insufficient evidence.
structured


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H04-C00', 'H02-C00'] | Temporary accounts require a support agent to reset the password.


The same results as a table, with the stamp line:

In [31]:
from shared.settings import chat_url, embed_url

# table(): the results as one table, with the stamp line
def table(results):
    print(f"{'chunking':22}{'chunks':>7}{'hit@3':>8}{'tokens per question':>21}{'policy answers right':>22}")
    for name, (n, hits, per_question, policy) in results.items():
        print(f"{name:22}{n:7}{hits:5}/30{per_question:21.0f}{policy:19}/10")
    print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}; {settings['CHAT_MODEL']} at {chat_url(settings)}")

table(results)

chunking               chunks   hit@3  tokens per question  policy answers right
passage                   125   30/30                  435                 10/10
window                    279   30/30                  256                 10/10
sentence                  622   30/30                  110                  0/10
structured                288   30/30                  228                 10/10

text-embedding-3-small at https://api.openai.com/v1; gpt-4o-mini at https://api.openai.com/v1


Every chunking scores hit@3 30 of 30, so the development questions cannot tell them apart. The tokens can: 435 a
question as whole paragraphs, 256 as windows, 228 as structured chunks and 110 as sentences. Chosen on those two
numbers, sentences win, and they are the only chunking that fails the policy questions, 0 of 10. When a whole-chunk row loses one reply of ten to *Insufficient evidence.*, as it can in your run, that is the abstention line's spread that Lab 02 measured, not the chunks.

This is the dead end of choosing by the development questions alone: written one fact at a time and naming their
subject, they reward small chunks. The policy questions cross a boundary. A measurement set needs both kinds before
it can choose.

### Checkpoint F

| You should see | Our run |
|---|---|
| hit@3 | 30/30 for all four |
| Tokens per question | passage 435, window 256, structured 228, sentence 110 |
| Policy answers right | passage 10/10, window 10/10, sentence 0/10, structured 10/10 |

---

## Part G: Add the title to each chunk (self-study)

Part G gives each chunk back some of the context its boundary cut away. A **context header** is a short text put in
front of a chunk before it is embedded and packed; the simplest is the document's title. The header goes in a field
of its own, `header`: `chunk_vectors()` embeds it in front of the text and `pack()` packs it there, while the text
stays the slice its offsets name. `tokens()` now counts the header too, since it is embedded and packed.

In [32]:
# with_title(): the same chunks with their title as a header; tokens() now counts the header too
def with_title(chunks):
    """The chunks with their title as a context header; the text is left as it was cut."""
    return [{**c, "header": f"{c['title'].replace('_', ' ')}: "} for c in chunks]

def tokens(chunks):
    """Tokens embedded and packed: the header, when a chunk has one, then the text."""
    return sum(count(c.get("header", "") + c["text"]) for c in chunks)

titled_sentences = with_title(CHUNKINGS["sentence"])
for c in titled_sentences:
    if c["chunk_id"] in ("H01-C01", "H04-C01", "S03-P01-C01"):
        print(c["chunk_id"], "|", c["header"] + c["text"])
print("lost characters:", lost_characters(titled_sentences, docs))

S03-P01-C01 | Nikola Tesla: He soon struck out on his own with financial backers, setting up laboratories and companies to develop a range of electrical devices.
H01-C01 | Atlas reset: Temporary accounts require a support agent.
H04-C01 | Boreal reset: Password reset uses a mobile app.
lost characters: 0


*Temporary accounts require a support agent.* now says *Atlas reset* first, and the mobile-app sentence says *Boreal
reset*, and the texts still slice back to their documents with nothing lost. Measure all four chunkings again with
their titles:

In [33]:
# titled: the four chunkings measured again with their titles
titled = {}
for name, chunks in CHUNKINGS.items():
    print(name + " + title")
    titled[name + " + title"] = measure(with_title(chunks))

passage + title


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C00'] | To reset the password on a temporary Atlas account, you need to contact a support agent.
window + title


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C00'] | To reset the password on a temporary Atlas account, you need to contact a support agent.
sentence + title


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C01'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H01-C01', 'H02-C00'] | To reset the password on a temporary Atlas account, you need to contact a support agent.
structured + title


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H02-C00', 'H04-C00'] | Atlas passwords can be reset by email, but temporary accounts require a support agent.


In [34]:
# both tables together, then the index size of each chunking without and with titles
table(results | titled)
for name, chunks in CHUNKINGS.items():
    print(f"index tokens, {name}: {tokens(chunks)} without titles, {tokens(with_title(chunks))} with")

chunking               chunks   hit@3  tokens per question  policy answers right
passage                   125   30/30                  435                 10/10
window                    279   30/30                  256                 10/10
sentence                  622   30/30                  110                  0/10
structured                288   30/30                  228                 10/10
passage + title           125   30/30                  446                 10/10
window + title            279   30/30                  266                 10/10
sentence + title          622   30/30                  122                 10/10
structured + title        288   30/30                  232                 10/10

text-embedding-3-small at https://api.openai.com/v1; gpt-4o-mini at https://api.openai.com/v1
index tokens, passage: 18867 without titles, 19337 with
index tokens, window: 21331 without titles, 22547 with
index tokens, sentence: 18915 without titles, 21330 with
index to

With titles every chunking gets 10 of 10 policy answers, the sentences included. For the temporary account, `H01-C01` is now in the top 3 and packed: its title gave it the word *Atlas* it was missing. For the first question the mobile-app sentence is still packed, but it now says *Boreal reset*, and the answer leaves it out. hit@3 stays 30 of 30 and the tokens rise a little. The titles add 470 tokens to the passage index and 2,415 to the sentence index, which has more chunks to label.

The handbook's paragraphs lost their headings in Part D. Give each one its heading as a header, as `with_title()` does, and measure them again:

In [35]:
# the handbook paragraphs with their heading as a header, measured on the 24 questions
report("paragraphs + heading", with_title(handbook_paragraphs))

paragraphs + heading    67 chunks | right first 23 of 24 | in the top 3 24 of 24 |   71 tokens per question
    rank 2: When can I reach a person?


With headings, the paragraphs still put the answering policy first for 23 of 24, for 10 more tokens a question (71);
the one miss moves from third to second. Each paragraph already names its topic, so the heading adds little; a header
matters where a chunk does not, as `H01-C01` showed.

**Which chunking to keep.** With titles every chunking passes all 32 questions, and sentences read the fewest tokens.
But the sentences found the exception only because its title matched the question's words, and they hand the answer
step one sentence, enough only because every question here needs one. Structured chunks keep a rule and its
exception together whatever the question says: for documents like these policies, choose structured chunks with
titles.

### Which chunks the pipeline keeps

The labs after this one index Lab 01's 272 window chunks and the five policies, each whole. The tables support that:
the policies are already whole, as structure-first splitting keeps them, and on the 30 development questions no
chunking beats the windows. In practice a chunking is chosen this way, frozen with the index, and changed only as a
measured change against it: a new baseline, never mixed with another change.

### Checkpoint G

| You should see | Our run |
|---|---|
| Titled chunks | `H01-C01` reads *Atlas reset: Temporary accounts require a support agent.*; lost characters: 0 |
| Policy answers right, with titles | 10/10 for all four, sentence + title included |
| Tokens per question, with titles | passage 446, window 266, sentence 122, structured 232 |
| Index tokens | passage 18,867 to 19,337; sentence 18,915 to 21,330; structured 18,882 to 19,992 |
| Handbook paragraphs with headings | first 23 of 24, top 3 24 of 24, 71 tokens a question |

---

## What you can do now

- Load a Markdown page and a PDF into records with their file, heading or page and offsets, and check that the
  sentences you know are in them came through word for word.
- Spot a table that lost its header in extraction, and write its rows as sentences that carry it.
- Cut documents into sentences or token windows, with each chunk's source ID and offsets, and check that no text was
  lost.
- Split a Markdown handbook at its headings and blank lines, and split by structure first under a token budget, with
  windows only for text that is too long.
- Compare chunkings on hit@3, tokens per question and questions that cross a boundary, and say why the easy
  questions alone chose wrong.
- Add a title header to each chunk and measure what it changes.

## Going further: LangChain's splitter and LLM-written context

**The same idea in LangChain.** `RecursiveCharacterTextSplitter`, where LangChain's documentation says to start for most
uses ([text splitters](https://docs.langchain.com/oss/python/integrations/splitters)), tries its separators
in order and keeps a piece whole while it fits. Two things to check before you use it: its default separators are
paragraph breaks, line breaks, spaces and single characters, with no sentence boundary, and its `chunk_size` counts
characters unless you build it from a tokenizer.

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# by_characters: a splitter sized in characters; by_tokens: the same splitter sized with the tokenizer
by_characters = RecursiveCharacterTextSplitter(chunk_size=96, chunk_overlap=0)
by_tokens = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=settings["TOKEN_ENCODING"], chunk_size=96, chunk_overlap=0, add_start_index=True)
print("Tesla, chunk_size in characters:", len(by_characters.split_text(tesla["text"])), "chunks")
for piece in by_tokens.create_documents([tesla["text"]]):
    print(piece.metadata, count(piece.page_content), "tokens |", piece.page_content[:40], "...", piece.page_content[-35:])

Tesla, chunk_size in characters: 7 chunks
{'start_index': 0} 96 tokens | Tesla gained experience in telephony and ... power development was involved in a
{'start_index': 510} 19 tokens | corporate alternating current/direct cur ...  as well as various patent battles.


Sized in characters, the splitter makes 7 chunks of at most 96 characters. Built from the tokenizer it makes two, 96
tokens and 19, and records each start in `start_index`, Part C's offset. Its first chunk ends inside a sentence, like
the window: no default separator marks a sentence end, so it fell back to spaces; `separators` takes your own list.

**Context written by a model.** Anthropic's [contextual retrieval](https://www.anthropic.com/news/contextual-retrieval)
asks a chat model, once per chunk, for a sentence that places the chunk in its document, and puts it in front of
the chunk. Anthropic reports fewer retrieval failures; that is the vendor's own measurement. Try it on the ten policy
sentences:

In [37]:
from shared.settings import chat_client, chat_options

# CONTEXT: the instruction to the chat model; contextual: the sentence chunks with a model-written header
CONTEXT = ("Write one short sentence that places the chunk within the document, so that a search can find the chunk."
           " Reply with that sentence only.")
texts = {doc["source_id"]: doc["text"] for doc in policies}
chat = chat_client(settings)
calls, billed, contextual = 0, 0, []
for c in policy_sentences:
    reply = chat.chat.completions.create(**chat_options("CHAT_MODEL", settings), messages=[
        {"role": "system", "content": CONTEXT},
        {"role": "user", "content": f"Document: {texts[c['source_id']]}\nChunk: {c['text']}"}])
    calls, billed = calls + 1, billed + reply.usage.total_tokens
    contextual.append({**c, "header": reply.choices[0].message.content.strip() + " "})
print(calls, "calls,", billed, "tokens |", contextual[1]["header"] + contextual[1]["text"])
print("right in all:", ask(contextual), "of 10")

10 calls, 880 tokens | The chunk "Temporary accounts require a support agent." is found in the document discussing Atlas password reset procedures. Temporary accounts require a support agent.


  right 5 of 5 | packed ['H01-C00', 'H01-C01', 'H02-C00'] | Atlas passwords can be reset by email.


  right 5 of 5 | packed ['H01-C00', 'H01-C01', 'H02-C00'] | To reset the password on a temporary Atlas account, you need to contact a support agent.
right in all: 10 of 10


Ten calls and about 880 tokens; the context for `H01-C01` names Atlas password resets. The policy answers are right 10 of 10, the same fix the title gave in Part G. The title cost no call; written context costs one call per chunk (622 for every sentence here), again whenever a document changes. Where titles already name each document's subject, measure the title first.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Keep the sentence before each sentence.** Overlap between windows repeats tokens wherever the cut falls. A
   different overlap repeats whole sentences: each chunk is a sentence together with the one before it.

   - Complete `with_previous()` under **Task 1**: chunk `n` runs from the start of sentence `n - 1` (or of sentence 0,
     for the first) to the end of sentence `n`. Build it from `sentences()` and `chunk()`.
   - Predict, before you run the cell: how many of the ten policy answers will be right, and how many tokens per
     question will the top 3 hold, against the sentence chunks in Part F?

   **Check:** `lost characters: 0`, then one row with chunks, hit@3, tokens per question and policy answers right.
   Our worked solution: 622 chunks, hit@3 30/30, 180 tokens per question and 10 of 10 policy answers right, with `H01-C01` packed for both questions.

   **Explain:** which policy chunk now carries what Part B's chunks lost? What does this overlap repeat that a title
   header does not?

2. **Optional: change the structure-first budget.** Part E used 96 tokens, the size of the windows. Build
   `structured()` chunkings with budgets of 48 and 200 under **Task 2**, and measure both.

   **Check:** two rows. Our worked solution: budget 48 gives 533 chunks and 108 tokens per question, budget 200 gives 147 chunks and 381; both 30/30 and 10/10.

   **Explain:** which number moved with the budget, and which did not? What kind of question would you add to the
   measurement set before choosing between them?

**Task 1.** Complete `with_previous()`, then run the cell:

In [38]:
def with_previous(doc):
    """Chunk n: from the start of sentence n - 1 (sentence 0 for the first) to the end of sentence n."""
    # TODO: s = sentences(doc), then one chunk(doc, n, start, end) for each sentence n
    return None

if with_previous(policies[0]) is None:
    print("⚠️ Complete with_previous(), then run the cell again.")
else:
    neighbours = [c for doc in docs for c in with_previous(doc)]
    print("lost characters:", lost_characters(neighbours, docs))
    table({"sentence + previous": measure(neighbours)})

⚠️ Complete with_previous(), then run the cell again.


**Task 2 (optional).** Two budgets, one row each:

In [39]:
BUDGETS = []   # TODO: the two budgets to try, 48 and 200

budget_results = {}
for budget in BUDGETS:
    print(f"structured, budget {budget}")
    budget_results[f"structured {budget}"] = measure([c for doc in docs for c in structured(doc, budget)])
if budget_results:
    table(budget_results)
else:
    print("⚠️ Put the two budgets in BUDGETS, then run the cell again.")

⚠️ Put the two budgets in BUDGETS, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: Unknown encoding o200k.` | `TOKEN_ENCODING` misspelt in `models.env` or `.env` | Use an encoding name `tiktoken` knows, such as `o200k_base`, then run Config again |
| `AssertionError: S03-P01-C01` in `lost_characters()` | a chunk's text was changed after it was cut (for example with `.strip()`), so its offsets no longer name it | Cut first and keep `text[start:end]`; put added text in another field, as `with_title()` puts the title in `header` |
| `BadRequestError: Error code: 400 ... Invalid 'input[0]': maximum input length is 8192 tokens.` | a chunk longer than the embedding model accepts, such as a document no splitter cut | Split it first; `structured()` keeps every chunk within its budget |
| LangChain's splitter returns many tiny chunks | `chunk_size` counts characters by default | Build it with `from_tiktoken_encoder(...)`, as in Going further |

## Resources

- [pypdf: extract text from a PDF](https://pypdf.readthedocs.io/en/stable/user/extract-text.html): `extract_text()`
  and its `layout` mode, used in Part A
- [OpenAI: tiktoken](https://github.com/openai/tiktoken): encodings, `encode` and `decode_with_offsets`
- [LangChain: text splitters](https://docs.langchain.com/oss/python/integrations/splitters): the recursive splitter
  and splitting by tokens
- [LangChain: split Markdown by headers](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter):
  `MarkdownHeaderTextSplitter`, Part D's split at headings with each heading kept as metadata
- [Chroma: evaluating chunking strategies for retrieval](https://www.trychroma.com/research/evaluating-chunking): a
  measured comparison of token, recursive and semantic chunkers
- [Qu et al., *Is semantic chunking worth the computational cost?*, NAACL Findings 2025](https://arxiv.org/abs/2410.13070):
  semantic splitting against fixed-size chunks
- [Anthropic: contextual retrieval](https://www.anthropic.com/news/contextual-retrieval): context written for each
  chunk before embedding, with the vendor's measurements

Next: [Lab 04](../04_Trace_And_Diagnose/README.md), *How to find the stage where a RAG answer failed*. Back to [the lab list](../README.md).